<a href="https://colab.research.google.com/github/Vargol/StableDiffusionColabs/blob/main/lokr_2_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import torch
from safetensors.torch import load_file, save_file
from pathlib import Path
import time


In [5]:
input_path = 'CHANGE THIS TO THE PATH OF A LOKR FILE YOU HAVE UPLOADED TO COLAB'
output_path = Path(input_path).stem + "_fixed_lora.safetensors"
target_rank = None

device = torch.device("cuda")

In [6]:
print(f"Loading {input_path}...")
state_dict = load_file(input_path)
new_state_dict = {}
groups = {}

for key, value in state_dict.items():
    if "lokr_" in key:
        base_key = key.replace(".lokr_w1_a", "").replace(".lokr_w1_b", "").replace(".lokr_w2_a", "").replace(".lokr_w2_b", "").replace(".lokr_w1", "").replace(".lokr_w2", "").replace(".alpha", "")
        if base_key not in groups:
            groups[base_key] = {}
        if "lokr_w1_a" in key:
            groups[base_key]["w1a"] = value
        elif "lokr_w1_b" in key:
            groups[base_key]["w1b"] = value
        elif "lokr_w2_a" in key:
            groups[base_key]["w2a"] = value
        elif "lokr_w2_b" in key:
            groups[base_key]["w2b"] = value
        elif "lokr_w1" in key:
            groups[base_key]["w1"] = value
        elif "lokr_w2" in key:
            groups[base_key]["w2"] = value
        elif "alpha" in key:
            groups[base_key]["alpha"] = value
    else:
        new_state_dict[key] = value

total_blocks = len(groups)
print(f"Found {total_blocks} LoKr blocks. Starting GPU conversion...")
start_time = time.time()
processed = 0

for base_key, parts in groups.items():
    processed += 1
    if ("w1" in parts or "w1a" in parts) and ("w2" in parts or "w2a" in parts):
        try:
            if "w1a" in parts:
                w1 = parts["w1a"] @ parts["w1b"]
                w1 = w1.to(device).float()
            else:
                w1 = parts["w1"].to(device).float()

            if "w2a" in parts:
                w2 = parts["w2a"] @ parts["w2b"]
                w2 = w2.to(device).float()
            else:
                w2 = parts["w2"].to(device).float()


            weight = torch.kron(w1, w2)

            try:
                u, s, vh = torch.linalg.svd(weight, full_matrices=False)
            except Exception as e:
                weight = weight.cpu()
                u, s, vh = torch.linalg.svd(weight, full_matrices=False)
                u, s, vh = u.to(device), s.to(device), vh.to(device)

            rank = target_rank if target_rank else min(w1.shape[0], w2.shape[1])

            u = u[:, :rank]
            s = s[:rank]
            vh = vh[:rank, :]

            sqrt_s = torch.diag(torch.sqrt(s))

            # DÜZELTME 1: YÖNLER DOĞRU BAĞLANDI
            # U * sqrt(S) = Çıkış Matrisi (lora_up) -> [out_dim, rank]
            # sqrt(S) * Vh = Giriş Matrisi (lora_down) -> [rank, in_dim]
            lora_up = torch.matmul(u, sqrt_s)
            lora_down = torch.matmul(sqrt_s, vh)

            new_key_base = base_key.replace("lokr_", "lora_")

            new_state_dict[f"{new_key_base}.lora_down.weight"] = lora_down.cpu().to(dtype=torch.float16)
            new_state_dict[f"{new_key_base}.lora_up.weight"] = lora_up.cpu().to(dtype=torch.float16)

            # DÜZELTME 2: 0 BOYUTLU (EMPTY) TENSOR HATASI GİDERİLDİ
            if "alpha" in parts:
                alpha_t = parts["alpha"].cpu()
                if alpha_t.dim() == 0:
                    alpha_t = alpha_t.unsqueeze(0) # Boş boyutu 1D array yapar (Draw Things çökmesini engeller)
                new_state_dict[f"{new_key_base}.alpha"] = alpha_t.to(dtype=torch.float16)
            else:
                new_state_dict[f"{new_key_base}.alpha"] = torch.tensor([rank], dtype=torch.float16)

            sys.stdout.write(f"\rConverting: {processed}/{total_blocks} blocks... ({(processed/total_blocks)*100:.1f}%)")
            sys.stdout.flush()

            del w1, w2, weight, u, s, vh, lora_down, lora_up
            if device.type == "mps":
                torch.mps.empty_cache()

        except Exception as e:
            print(f"\nError converting block {base_key}: {e}")
            continue

save_file(new_state_dict, output_path)
print(f"\nDone! Total time: {time.time() - start_time:.2f} seconds.")

Loading /content/SNOFS_V13D_KREA2_%5BAshen3%5D.safetensors...
Found 256 LoKr blocks. Starting GPU conversion...
Converting: 256/256 blocks... (100.0%)
Done! Total time: 7099.82 seconds.
